# 09 SIPP：安全间隔上的 A*


本课按百科条目写。SIPP 用「空闲时间段」代替每一个时刻，状态更少，最优性仍在。调度台在最后。

---

## 1. 定义

**Safe Interval Path Planning**（Phillips & Likhachev, ICRA 2011）：不把每个整数时刻当状态，而把每个点被别人挖洞之后剩下的空闲时间收成区间 \([lo,hi)\)。

状态 = (点, 第几个空闲区间)。区间内何时到达只记一个到达时刻 `arr`（同一区间内更晚到达绝不会更好——非负等待）。

- 输入：图、起终点、占用表、horizon。
- 输出：与时间扩展 A\* 相同代价的路径（空表时扩展接近普通 A\*）。

---

## 2. 为何比时间扩展更省

时间扩展：状态 \(O(|V|T)\)。  
SIPP：每点区间数 ≤ 占用次数 + 1，通常远小于 T。空表时每点一个区间 `[0,T)`，行为接近空间 A\*（仍允许在区间内等待，但等不会单独产生新状态）。

有交通时，SIPP 扩展应 **少于** 时间 A\*，代价相同。

---

## 3. 不变量

- 到达时刻落在所选区间：`lo ≤ arr < hi`。
- 进入邻居：`arr' = max(arr + step, nlo)`，不能早于邻居区间开门；且 `arr' < nhi`。
- 边在离开拍 `arr'-1`（本课 step=1 时即 `earliest-1`）必须空闲。
- 空占用表时代价 = 时间扩展 A\* = 空间最短路。

---

## 4. 伪代码

```
intervals[v] ← SafeIntervals(v)
选包含 t=0 的起点区间 i0
g[s,i0] ← 0
push (h(s), 0, s, i0)
while heap:
    (f, arr, u, i) ← pop
    if (u,i) 已关闭: continue
    关闭
    if u = goal: return
    [lo,hi] ← intervals[u][i]
    for v in 邻居(u):          # 等待已编码在区间内，不必单独等
        move ← arr + 1
        for 区间 j = [nlo, nhi] of v:
            if move ≥ nhi: continue
            earliest ← max(move, nlo)
            if earliest ≥ nhi: continue
            若边在 earliest-1 被占: continue
            松弛 (v,j) 的到达 earliest
```

`safe_intervals`：占用时刻排序，相邻占用之间的空隙即区间。半开。

---

## 5. 手算区间

占用 {1,2,5}，horizon=8，别人占的：

```
t:  0 1 2 3 4 5 6 7
    . # # . . # . .
```

切开：`[0,1)`, `[3,5)`, `[6,8)`。

从某点 t=0 出发，一步到该点：move=1，但 1 落在占用里，不能进第一段；最早 `max(1,3)=3` 进第二段。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：先写区间，再写 SIPP 主循环


In [ ]:
from lib.maps import corridor_headon
from lib.reservation import ReservationTable


def safe_intervals_mini(blocked_times, horizon):
    # blocked_times: 别人占用该点的时刻
    blocked = sorted(t for t in blocked_times if t < horizon)
    intervals = []
    start = 0
    for t in blocked:
        if t > start:
            intervals.append((start, t))
        start = t + 1
    if start < horizon:
        intervals.append((start, horizon))
    return intervals


print("占用 {1,2,5} horizon=8 ->", safe_intervals_mini({1, 2, 5}, 8))
print("空 ->", safe_intervals_mini(set(), 8))


In [ ]:
import heapq
from lib.search import astar
from lib.algos.temporal import time_astar, sipp


def sipp_mini(graph, start, goal, table, agent="me", horizon=40, verbose=True):
    cache = {}

    def ivals(n):
        if n not in cache:
            cache[n] = table.safe_intervals(n, horizon, agent)
        return cache[n]

    start_iv = None
    for i, (lo, hi) in enumerate(ivals(start)):
        if lo <= 0 < hi:
            start_iv = i
            break
    if start_iv is None:
        print("起点 t=0 不空闲")
        return None, float("inf"), 0

    heap = []
    g = {(start, start_iv): 0}
    came = {(start, start_iv): None}
    heapq.heappush(heap, (graph.heuristic(start, goal), 0, start, start_iv))
    seen = set()
    expanded = 0
    while heap:
        _f, arr, node, ii = heapq.heappop(heap)
        state = (node, ii)
        if state in seen:
            continue
        seen.add(state)
        expanded += 1
        if verbose:
            lo, hi = ivals(node)[ii]
            print(f"关闭 {node:3} 区间{ii}{ivals(node)[ii]} 到达={arr}")
        if node == goal:
            chain = []
            cur = state
            while cur is not None:
                chain.append(cur[0])
                cur = came[cur]
            chain.reverse()
            if verbose:
                print("节点列", chain, "到达", arr, "扩展", expanded)
            return chain, float(arr), expanded
        for nxt, _step in graph.neighbors(node):
            move_t = arr + 1
            for jj, (nlo, nhi) in enumerate(ivals(nxt)):
                if move_t >= nhi:
                    continue
                earliest = max(move_t, nlo)
                if earliest >= nhi:
                    continue
                if not table.edge_free(node, nxt, earliest - 1, agent):
                    continue
                ns = (nxt, jj)
                if earliest < g.get(ns, float("inf")):
                    g[ns] = earliest
                    came[ns] = state
                    heapq.heappush(heap, (earliest + graph.heuristic(nxt, goal), earliest, nxt, jj))
    return None, float("inf"), expanded


g = corridor_headon()
table = ReservationTable()
print("空表 C2", table.safe_intervals("C2", 20, "me"))
for t in range(3):
    table.occupy("C2", t, "blk")
print("占 0,1,2 后 C2", table.safe_intervals("C2", 20, "me"))

print("--- SIPP ---")
sp, sc, se = sipp_mini(g, "A", "B", table)
print("--- 对照 ---")
ta = time_astar(g, "A", "B", table=table, agent="me")
si = sipp(g, "A", "B", table=table, agent="me")
print("本课", sc, "扩展", se)
print("timeA*", ta.cost, "扩展", ta.expanded)
print("库SIPP", si.cost, "扩展", si.expanded)
print("空表时 SIPP 应接近空间 A*", astar(g, "A", "B", record=False).cost)


## 7. 逐行

| 块 | 作用 |
|----|------|
| `ivals(n)` | 惰性缓存该点区间 |
| 起点选含 0 的区间 | 否则无解 |
| 双重循环 邻居×区间 | 尝试插入 |
| `earliest=max(move,nlo)` | 早到则等到区间开门 |
| 键 `earliest+h(v)` | 仍是 A\* |
| 等待 | 不单独产生状态，早到等于在区间开头等 |

有占用时 SIPP 扩展应 ≤ 时间 A\*，代价相同。

## 常见 bug

1. 区间用闭区间，占用拍被当成空闲。
2. 忘记边在 `earliest-1` 的换位检查。
3. 把区间编号和到达时刻当成同一个数。
4. 起点 t=0 已被占却没处理「无起始区间」。
5. 回溯只记下节点、没把等待展开：执行层时间对不齐。教学版库返回的是节点列，到达时刻在 cost 里。

## 条目小结

| 项目 | 内容 |
|------|------|
| 状态 | (点, 区间编号) |
| 相对时间 A\* | 状态少，代价同 |
| 空表 | 接近普通 A\* |
| 下一课 | 多车：联合状态 vs 优先规划 |

## 练习

1. 占用 {1,2,5}、horizon=8，手写区间，与 `safe_intervals_mini` 对照。
2. 空表跑 SIPP 与时间 A\*，扩展谁更大？为什么 SIPP 往往更小？
3. `earliest = max(move, nlo)` 对应「等到开门」。若写成 `move` 不管 nlo，会怎样？


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.algos.temporal import sipp

si = sipp(g, "A", "B", table=table, agent="me")
studio = PlannerStudio()
studio.show_search(g.to_studio(), si, "SIPP")
studio
